# AFLW2000-3D Cropped Yaw-Only Evaluation

This notebook benchmarks yaw estimation on the 3DDFA-preprocessed AFLW2000-3D crop set. The available benchmark files are `AFLW2000-3D_crop.list` and `AFLW2000-3D.pose.npy`, so the evaluation is yaw-only. Pitch and roll are still emitted by the model adapters for diagnostics, but the benchmark metrics only use yaw.

In [1]:
from __future__ import annotations

import json
import shutil
import sys
import time
import urllib.request
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Optional

import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
import pandas as pd
from IPython.display import display


def find_workspace_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "openwillis-face" / "src").exists() and (base / "head_pose_eval").exists():
            return base
    return Path.cwd()


WORKSPACE_ROOT = find_workspace_root()
BENCHMARK_ROOT = WORKSPACE_ROOT / "head_pose_eval"
OPENWILLIS_SRC = WORKSPACE_ROOT / "openwillis-face" / "src"
if OPENWILLIS_SRC.exists() and str(OPENWILLIS_SRC) not in sys.path:
    sys.path.insert(0, str(OPENWILLIS_SRC))

DATASET_SEARCH_START = WORKSPACE_ROOT
OUTPUTS_DIR = BENCHMARK_ROOT / "outputs"
MODEL_PATH = WORKSPACE_ROOT / "head_movement" / "models" / "face_landmarker.task"
METHODS = ["pyfeat_img2pose", "owf_head_movement", "mediapipe_matrix", "mediapipe_pnp"]
LIMIT = 10  # set to None for the full benchmark
EXPECTED_COUNT = 2000

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (10, 4)

print("WORKSPACE_ROOT:", WORKSPACE_ROOT)
print("OPENWILLIS_SRC:", OPENWILLIS_SRC, OPENWILLIS_SRC.exists())
print("MODEL_PATH:", MODEL_PATH, MODEL_PATH.exists())
print("OUTPUTS_DIR:", OUTPUTS_DIR)

WORKSPACE_ROOT: /Users/pelmeshek1706/Desktop/projects/airest-face
OPENWILLIS_SRC: /Users/pelmeshek1706/Desktop/projects/airest-face/openwillis-face/src True
MODEL_PATH: /Users/pelmeshek1706/Desktop/projects/airest-face/head_movement/models/face_landmarker.task True
OUTPUTS_DIR: /Users/pelmeshek1706/Desktop/projects/airest-face/head_pose_eval/outputs


## 1. Dataset Loader

In [2]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
GROUND_TRUTH_COLUMNS = ["image_id", "image_index", "image_path", "gt_yaw", "abs_gt_yaw", "yaw_bucket"]
PREDICTION_COLUMNS = ["image_id", "method", "pred_pitch", "pred_yaw", "pred_roll", "success", "failure_reason", "runtime_ms"]


def yaw_bucket(gt_yaw: float) -> str:
    abs_yaw = abs(float(gt_yaw))
    if abs_yaw <= 30:
        return "<=30"
    if abs_yaw <= 60:
        return "30-60"
    return ">60"


def _candidate_layout_roots(start: Path) -> list[Path]:
    start = Path(start).expanduser()
    bases = [start, *start.parents]
    relative_roots = [
        Path('.'),
        Path('datasets'),
        Path('head_pose_eval'),
        Path('head_pose_eval/datasets'),
        Path('head_pose_eval/notebooks/head_pose_eval'),
        Path('head_pose_eval/notebooks/head_pose_eval/datasets'),
        Path('notebooks/head_pose_eval'),
        Path('notebooks/head_pose_eval/datasets'),
    ]

    candidates: list[Path] = []
    seen: set[Path] = set()
    for base in bases:
        for rel in relative_roots:
            candidate = (base / rel).resolve()
            if candidate in seen:
                continue
            seen.add(candidate)
            candidates.append(candidate)
    return candidates


def resolve_dataset_layout(start: Path) -> tuple[Path, Path, Path]:
    for root in _candidate_layout_roots(start):
        for prefix in (Path('datasets'), Path('.')):
            data_root = root / prefix / 'test.data'
            config_root = root / prefix / 'test.configs'
            image_dir = data_root / 'AFLW2000-3D_crop'
            list_path = data_root / 'AFLW2000-3D_crop.list'
            gt_yaw_path = config_root / 'AFLW2000-3D.pose.npy'
            if image_dir.exists() and list_path.exists() and gt_yaw_path.exists():
                return image_dir, list_path, gt_yaw_path

    raise FileNotFoundError(
        'Could not locate the 3DDFA AFLW2000-3D crop benchmark. Expected image crops, list file, and AFLW2000-3D.pose.npy under test.data/test.configs.'
    )


def _resolve_image_path(image_dir: Path, token: str) -> Path:
    name = Path(token).name
    candidate = image_dir / name
    if candidate.exists():
        return candidate

    stem = Path(name).stem
    suffix = Path(name).suffix.lower()
    if not suffix:
        for ext in IMAGE_EXTENSIONS:
            candidate = image_dir / f"{stem}{ext}"
            if candidate.exists():
                return candidate
    else:
        for ext in IMAGE_EXTENSIONS:
            candidate = image_dir / f"{stem}{ext}"
            if candidate.exists():
                return candidate

    raise FileNotFoundError(f"Could not resolve image path for list token: {token}")


def build_ground_truth_index(dataset_root: Path, expected_count: int = 2000) -> pd.DataFrame:
    image_dir, list_path, gt_yaw_path = resolve_dataset_layout(dataset_root)

    raw_lines = [line.strip() for line in list_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    if not raw_lines:
        raise FileNotFoundError(f"No entries found in {list_path}")

    gt_yaw = np.asarray(np.load(gt_yaw_path, allow_pickle=True)).reshape(-1).astype(float)
    if len(raw_lines) != len(gt_yaw):
        raise ValueError(f"List length and yaw length differ: {len(raw_lines)} != {len(gt_yaw)}")

    records: list[dict[str, object]] = []
    for index, (line, yaw) in enumerate(zip(raw_lines, gt_yaw)):
        image_path = _resolve_image_path(image_dir, line.split()[0])
        records.append(
            {
                'image_id': image_path.stem,
                'image_index': index,
                'image_path': str(image_path),
                'gt_yaw': float(yaw),
                'abs_gt_yaw': abs(float(yaw)),
                'yaw_bucket': yaw_bucket(float(yaw)),
            }
        )

    df = pd.DataFrame(records, columns=GROUND_TRUTH_COLUMNS)
    if df['image_id'].duplicated().any():
        duplicates = df.loc[df['image_id'].duplicated(), 'image_id'].head().tolist()
        raise ValueError(f"Duplicate image_id values found: {duplicates}")

    if expected_count and len(df) != expected_count:
        warnings.warn(
            f"Expected {expected_count} AFLW2000-3D crop images, found {len(df)}.",
            RuntimeWarning,
            stacklevel=2,
        )

    return df

## 2. Pose Helpers and Method Adapters

In [3]:
FACE_LANDMARKER_URL = (
    'https://storage.googleapis.com/mediapipe-models/face_landmarker/'
    'face_landmarker/float16/1/face_landmarker.task'
)

PNP_LANDMARK_IDS = {
    'nose_tip': 1,
    'chin': 152,
    'left_eye_outer': 33,
    'right_eye_outer': 263,
    'left_mouth': 61,
    'right_mouth': 291,
}

GENERIC_FACE_MODEL_POINTS = np.array(
    [
        [0.0, 0.0, 0.0],
        [0.0, -63.6, -12.5],
        [-43.3, 32.7, -26.0],
        [43.3, 32.7, -26.0],
        [-28.9, -28.9, -24.1],
        [28.9, -28.9, -24.1],
    ],
    dtype=np.float64,
)


@dataclass
class PosePrediction:
    pred_pitch: float = np.nan
    pred_yaw: float = np.nan
    pred_roll: float = np.nan
    success: bool = False
    failure_reason: Optional[str] = None


class PoseMethod:
    method_id: str

    def predict(self, image_bgr: np.ndarray) -> PosePrediction:
        raise NotImplementedError

    def close(self) -> None:
        return None


def timed_prediction(method: PoseMethod, image_id: str, image_bgr: np.ndarray) -> dict[str, object]:
    start_time = time.perf_counter()
    try:
        prediction = method.predict(image_bgr)
    except Exception as exc:
        prediction = PosePrediction(success=False, failure_reason=f"{type(exc).__name__}: {exc}")
    runtime_ms = (time.perf_counter() - start_time) * 1000.0
    return {
        'image_id': image_id,
        'method': method.method_id,
        'pred_pitch': prediction.pred_pitch,
        'pred_yaw': prediction.pred_yaw,
        'pred_roll': prediction.pred_roll,
        'success': bool(prediction.success),
        'failure_reason': prediction.failure_reason,
        'runtime_ms': runtime_ms,
    }


def _finite_pose(pitch: float, yaw: float, roll: float) -> PosePrediction:
    values = np.array([pitch, yaw, roll], dtype=float)
    if not np.isfinite(values).all():
        return PosePrediction(success=False, failure_reason='non_finite_pose')
    return PosePrediction(float(pitch), float(yaw), float(roll), True, None)


def orthonormalize_rotation(matrix: np.ndarray) -> np.ndarray:
    u, _, vh = np.linalg.svd(matrix)
    rotation = u @ vh
    if np.linalg.det(rotation) < 0:
        u[:, -1] *= -1.0
        rotation = u @ vh
    return rotation


def rotation_matrix_to_pitch_roll_yaw(rotation_matrix: np.ndarray) -> np.ndarray:
    rotation_matrix = orthonormalize_rotation(np.asarray(rotation_matrix, dtype=np.float64))
    sy = np.clip(rotation_matrix[0, 2], -1.0, 1.0)
    yaw = np.arcsin(sy)
    cy = np.cos(yaw)
    if abs(cy) > 1e-6:
        pitch = np.arctan2(-rotation_matrix[1, 2], rotation_matrix[2, 2])
        roll = np.arctan2(-rotation_matrix[0, 1], rotation_matrix[0, 0])
    else:
        pitch = np.arctan2(rotation_matrix[2, 1], rotation_matrix[1, 1])
        roll = 0.0
    return np.degrees([pitch, roll, yaw]).astype(float)


def ensure_face_landmarker_task(model_path: Path, model_url: str = FACE_LANDMARKER_URL) -> Path:
    model_path = Path(model_path)
    model_path.parent.mkdir(parents=True, exist_ok=True)
    if not model_path.exists():
        urllib.request.urlretrieve(model_url, model_path)
    return model_path


def create_face_landmarker(model_path: Path):
    base_options = mp.tasks.BaseOptions(model_asset_path=str(model_path))
    options = mp.tasks.vision.FaceLandmarkerOptions(
        base_options=base_options,
        running_mode=mp.tasks.vision.RunningMode.IMAGE,
        num_faces=1,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=True,
    )
    return mp.tasks.vision.FaceLandmarker.create_from_options(options)


def _mediapipe_image(image_bgr: np.ndarray) -> mp.Image:
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)


def _landmarks_to_numpy(face_landmarks) -> np.ndarray:
    return np.array([[landmark.x, landmark.y, landmark.z] for landmark in face_landmarks], dtype=np.float64)


def solve_pnp_rotation(landmarks_xyz: np.ndarray, width: int, height: int) -> Optional[np.ndarray]:
    image_points = []
    for landmark_id in PNP_LANDMARK_IDS.values():
        image_points.append([landmarks_xyz[landmark_id, 0] * width, landmarks_xyz[landmark_id, 1] * height])
    image_points = np.array(image_points, dtype=np.float64)

    camera_matrix = np.array(
        [[width, 0.0, width / 2.0], [0.0, width, height / 2.0], [0.0, 0.0, 1.0]],
        dtype=np.float64,
    )
    success, rvec, _ = cv2.solvePnP(
        GENERIC_FACE_MODEL_POINTS,
        image_points,
        camera_matrix,
        np.zeros((4, 1), dtype=np.float64),
        flags=cv2.SOLVEPNP_ITERATIVE,
    )
    if not success:
        return None
    rotation_matrix, _ = cv2.Rodrigues(rvec)
    return orthonormalize_rotation(rotation_matrix)


class PyFeatImg2PoseMethod(PoseMethod):
    method_id = 'pyfeat_img2pose'

    def __init__(self) -> None:
        import feat

        self.detector = feat.Detector()

    def predict(self, image_bgr: np.ndarray) -> PosePrediction:
        faceposes = self.detector.detect_facepose(image_bgr)
        poses = faceposes['poses'][0]
        if len(poses) == 0:
            return PosePrediction(success=False, failure_reason='no_face')
        pitch, roll, yaw = poses[0]
        return _finite_pose(pitch, yaw, roll)


class OWFHeadMovementMethod(PoseMethod):
    method_id = 'owf_head_movement'

    def __init__(self) -> None:
        import feat
        from openwillis.face.head_movement import get_facepose

        self.detector = feat.Detector()
        self.get_facepose: Callable[[int, np.ndarray, object], np.ndarray] = get_facepose

    def predict(self, image_bgr: np.ndarray) -> PosePrediction:
        row = self.get_facepose(0, image_bgr, self.detector)
        pitch, roll, yaw = row[6], row[7], row[8]
        if np.isnan([pitch, roll, yaw]).any():
            return PosePrediction(success=False, failure_reason='no_face')
        return _finite_pose(pitch, yaw, roll)


class MediaPipeMatrixMethod(PoseMethod):
    method_id = 'mediapipe_matrix'

    def __init__(self, model_path: Path) -> None:
        self.landmarker = create_face_landmarker(ensure_face_landmarker_task(model_path))

    def predict(self, image_bgr: np.ndarray) -> PosePrediction:
        result = self.landmarker.detect(_mediapipe_image(image_bgr))
        if len(result.face_landmarks) != 1:
            return PosePrediction(success=False, failure_reason=f'face_count={len(result.face_landmarks)}')
        if len(result.facial_transformation_matrixes) != 1:
            return PosePrediction(success=False, failure_reason='missing_transformation_matrix')
        matrix4x4 = np.asarray(result.facial_transformation_matrixes[0], dtype=np.float64).reshape(4, 4)
        pitch, roll, yaw = rotation_matrix_to_pitch_roll_yaw(matrix4x4[:3, :3])
        return _finite_pose(pitch, yaw, roll)

    def close(self) -> None:
        self.landmarker.close()


class MediaPipePnPMethod(PoseMethod):
    method_id = 'mediapipe_pnp'

    def __init__(self, model_path: Path) -> None:
        self.landmarker = create_face_landmarker(ensure_face_landmarker_task(model_path))

    def predict(self, image_bgr: np.ndarray) -> PosePrediction:
        height, width = image_bgr.shape[:2]
        result = self.landmarker.detect(_mediapipe_image(image_bgr))
        if len(result.face_landmarks) != 1:
            return PosePrediction(success=False, failure_reason=f'face_count={len(result.face_landmarks)}')
        landmarks_xyz = _landmarks_to_numpy(result.face_landmarks[0])
        rotation = solve_pnp_rotation(landmarks_xyz, width, height)
        if rotation is None:
            return PosePrediction(success=False, failure_reason='solvepnp_failed')
        pitch, roll, yaw = rotation_matrix_to_pitch_roll_yaw(rotation)
        return _finite_pose(pitch, yaw, roll)

    def close(self) -> None:
        self.landmarker.close()


def create_method(method_id: str, model_path: Path = Path('head_movement/models/face_landmarker.task')) -> PoseMethod:
    if method_id == 'pyfeat_img2pose':
        return PyFeatImg2PoseMethod()
    if method_id == 'owf_head_movement':
        return OWFHeadMovementMethod()
    if method_id == 'mediapipe_matrix':
        return MediaPipeMatrixMethod(model_path)
    if method_id == 'mediapipe_pnp':
        return MediaPipePnPMethod(model_path)
    raise ValueError(f'Unknown method_id: {method_id}')


METHOD_IDS = ['pyfeat_img2pose', 'owf_head_movement', 'mediapipe_matrix', 'mediapipe_pnp']

## 3. Prediction Runner

In [4]:
def run_method_on_ground_truth(
    ground_truth_df: pd.DataFrame,
    method_id: str,
    model_path: Path,
) -> pd.DataFrame:
    method = create_method(method_id, model_path=model_path)
    rows: list[dict[str, object]] = []
    try:
        for record in ground_truth_df.itertuples(index=False):
            image_bgr = cv2.imread(record.image_path)
            if image_bgr is None:
                rows.append(
                    {
                        'image_id': record.image_id,
                        'method': method_id,
                        'pred_pitch': float('nan'),
                        'pred_yaw': float('nan'),
                        'pred_roll': float('nan'),
                        'success': False,
                        'failure_reason': 'image_read_failed',
                        'runtime_ms': 0.0,
                    }
                )
                continue
            rows.append(timed_prediction(method, record.image_id, image_bgr))
    finally:
        method.close()

    return pd.DataFrame(rows, columns=PREDICTION_COLUMNS)


def run_raw_predictions(
    ground_truth_df: pd.DataFrame,
    methods: list[str],
    outputs_dir: Path,
    model_path: Path,
) -> pd.DataFrame:
    outputs_dir.mkdir(parents=True, exist_ok=True)
    raw_dir = outputs_dir / 'raw_predictions'
    if raw_dir.exists():
        shutil.rmtree(raw_dir)
    raw_dir.mkdir(parents=True, exist_ok=True)

    prediction_frames = []
    for method_id in methods:
        predictions = run_method_on_ground_truth(ground_truth_df, method_id, model_path=model_path)
        predictions.to_csv(raw_dir / f'{method_id}_predictions.csv', index=False)
        prediction_frames.append(predictions)

    if not prediction_frames:
        return pd.DataFrame(columns=PREDICTION_COLUMNS)
    return pd.concat(prediction_frames, ignore_index=True)

## 4. Yaw Sign Normalization

In [5]:
def build_review_subset(ground_truth_df: pd.DataFrame, per_bucket: int = 15, random_state: int = 17) -> pd.DataFrame:
    df = ground_truth_df.copy()
    df['yaw_bucket'] = df['gt_yaw'].map(yaw_bucket)
    parts = []
    for bucket in ['<=30', '30-60', '>60']:
        bucket_df = df[df['yaw_bucket'] == bucket]
        if len(bucket_df) == 0:
            continue
        parts.append(bucket_df.sample(min(per_bucket, len(bucket_df)), random_state=random_state))
    return pd.concat(parts, ignore_index=True) if parts else df.head(0)


def candidate_yaw_signs() -> dict[str, int]:
    return {'same': 1, 'invert_yaw': -1}


def apply_yaw_sign(values: np.ndarray, sign: int) -> np.ndarray:
    return values * float(sign)


def load_raw_predictions(raw_predictions_dir: Path) -> pd.DataFrame:
    paths = sorted(Path(raw_predictions_dir).glob('*_predictions.csv'))
    if not paths:
        raise FileNotFoundError(f'No prediction CSV files found in {raw_predictions_dir}')
    return pd.concat([pd.read_csv(path) for path in paths], ignore_index=True)


def mapping_metrics(predictions_df: pd.DataFrame, review_subset_df: pd.DataFrame) -> pd.DataFrame:
    joined = review_subset_df[['image_id', 'gt_yaw']].merge(predictions_df, on='image_id')
    joined = joined[joined['success']].dropna(subset=['pred_yaw', 'gt_yaw'])

    rows = []
    candidates = candidate_yaw_signs()
    for method in sorted(joined['method'].unique()):
        method_rows = joined[joined['method'] == method]
        gt = method_rows['gt_yaw'].to_numpy(dtype=float)
        pred = method_rows['pred_yaw'].to_numpy(dtype=float)
        for name, sign in candidates.items():
            mapped = apply_yaw_sign(pred, sign)
            error = mapped - gt
            abs_error = np.abs(error)
            rows.append(
                {
                    'method': method,
                    'mapping': name,
                    'yaw_sign': int(sign),
                    'review_count': len(method_rows),
                    'yaw_mae': float(abs_error.mean()) if len(abs_error) else np.nan,
                    'yaw_median_ae': float(np.median(abs_error)) if len(abs_error) else np.nan,
                    'yaw_p90_ae': float(np.quantile(abs_error, 0.90)) if len(abs_error) else np.nan,
                    'yaw_signed_bias': float(error.mean()) if len(error) else np.nan,
                }
            )

    if not rows:
        return pd.DataFrame(columns=['method', 'mapping', 'yaw_sign', 'review_count', 'yaw_mae', 'yaw_median_ae', 'yaw_p90_ae', 'yaw_signed_bias'])
    return pd.DataFrame(rows).sort_values(['method', 'yaw_mae'])


def derive_mapping_config(mapping_metrics_df: pd.DataFrame) -> dict[str, dict[str, object]]:
    config: dict[str, dict[str, object]] = {}
    for method, group in mapping_metrics_df.groupby('method'):
        best = group.sort_values('yaw_mae').iloc[0]
        config[method] = {
            'name': str(best['mapping']),
            'yaw_sign': int(best['yaw_sign']),
        }
    return config


def apply_method_mappings(predictions_df: pd.DataFrame, mapping_config: dict[str, dict[str, object]]) -> pd.DataFrame:
    normalized = predictions_df.copy()
    for method, mapping in mapping_config.items():
        mask = normalized['method'] == method
        if not mask.any():
            continue
        sign = int(mapping['yaw_sign'])
        normalized.loc[mask, 'pred_yaw'] = apply_yaw_sign(normalized.loc[mask, 'pred_yaw'].to_numpy(dtype=float), sign)
    return normalized


def write_convention_review(
    output_path: Path,
    mapping_metrics_df: pd.DataFrame,
    mapping_config: dict[str, dict[str, object]],
) -> None:
    lines = [
        '# Convention Review',
        '',
        'This benchmark only normalizes the yaw sign. Pitch and roll are kept as diagnostics.',
        'The locked sign is derived from a deterministic review subset, not from the full benchmark.',
        '',
        '## Locked Yaw Sign Candidates',
        '',
    ]
    for method, mapping in mapping_config.items():
        lines.append(f"- `{method}`: `{mapping['name']}` yaw_sign={mapping['yaw_sign']}")

    lines.extend(['', '## Candidate Metrics Per Method', ''])
    for method, group in mapping_metrics_df.groupby('method'):
        lines.append(f'### {method}')
        lines.append('```text')
        lines.append(group.to_string(index=False))
        lines.append('```')
        lines.append('')

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text('\n'.join(lines), encoding='utf-8')


def write_review_contact_sheet(
    review_subset_df: pd.DataFrame,
    output_path: Path,
    thumb_width: int = 180,
) -> None:
    if review_subset_df.empty:
        return
    thumbs = []
    for record in review_subset_df.itertuples(index=False):
        image = cv2.imread(record.image_path)
        if image is None:
            continue
        scale = thumb_width / image.shape[1]
        thumb = cv2.resize(image, (thumb_width, max(1, int(image.shape[0] * scale))))
        label = f"{record.image_id} y={record.gt_yaw:.1f} {record.yaw_bucket}"
        cv2.putText(thumb, label, (5, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (0, 255, 255), 1, cv2.LINE_AA)
        thumbs.append(thumb)
    if not thumbs:
        return

    cols = 5
    rows = int(np.ceil(len(thumbs) / cols))
    cell_h = max(thumb.shape[0] for thumb in thumbs)
    sheet = np.full((rows * cell_h, cols * thumb_width, 3), 255, dtype=np.uint8)
    for idx, thumb in enumerate(thumbs):
        row, col = divmod(idx, cols)
        y0 = row * cell_h
        x0 = col * thumb_width
        sheet[y0 : y0 + thumb.shape[0], x0 : x0 + thumb.shape[1]] = thumb
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output_path), sheet)


def normalize_predictions(
    ground_truth_df: pd.DataFrame,
    raw_predictions_dir: Path,
    outputs_dir: Path,
    mapping_config_path: Path | None = None,
) -> tuple[pd.DataFrame, dict[str, dict[str, object]], pd.DataFrame]:
    outputs_dir.mkdir(parents=True, exist_ok=True)
    metrics_dir = outputs_dir / 'metrics'
    plots_dir = outputs_dir / 'plots'
    metrics_dir.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)

    predictions_df = load_raw_predictions(raw_predictions_dir)
    review_subset_df = build_review_subset(ground_truth_df)
    metrics_df = mapping_metrics(predictions_df, review_subset_df)
    metrics_df.to_csv(metrics_dir / 'convention_candidate_metrics.csv', index=False)

    if mapping_config_path and Path(mapping_config_path).exists():
        mapping_config = json.loads(Path(mapping_config_path).read_text(encoding='utf-8'))
    else:
        mapping_config = derive_mapping_config(metrics_df)
        for method in sorted(predictions_df['method'].dropna().unique()):
            mapping_config.setdefault(method, {'name': 'same', 'yaw_sign': 1})

    (metrics_dir / 'convention_mapping.json').write_text(json.dumps(mapping_config, indent=2), encoding='utf-8')
    write_convention_review(metrics_dir / 'convention_review.md', metrics_df, mapping_config)
    write_review_contact_sheet(review_subset_df, plots_dir / 'convention_review_contact_sheet.jpg')

    normalized = apply_method_mappings(predictions_df, mapping_config)
    normalized.to_csv(outputs_dir / 'normalized_predictions.csv', index=False)
    return normalized, mapping_config, metrics_df

## 5. Yaw Metrics

In [6]:
def joined_success_rows(ground_truth_df: pd.DataFrame, predictions_df: pd.DataFrame) -> pd.DataFrame:
    joined = predictions_df.merge(ground_truth_df, on='image_id', how='left')
    joined = joined[joined['success']].dropna(subset=['gt_yaw', 'pred_yaw'])
    joined['yaw_error'] = joined['pred_yaw'] - joined['gt_yaw']
    joined['abs_yaw_error'] = joined['yaw_error'].abs()
    return joined


def compute_metrics_summary(ground_truth_df: pd.DataFrame, predictions_df: pd.DataFrame) -> pd.DataFrame:
    total_images = ground_truth_df['image_id'].nunique()
    joined = predictions_df.merge(ground_truth_df, on='image_id', how='left')
    success_rows = joined_success_rows(ground_truth_df, predictions_df)
    rows = []

    for method, method_all in joined.groupby('method'):
        method_success = success_rows[success_rows['method'] == method]
        rows.append(
            {
                'method': method,
                'image_count': total_images,
                'success_count': int(method_all['success'].sum()),
                'failure_count': int((~method_all['success'].astype(bool)).sum()),
                'detection_rate': float(method_all['success'].mean()),
                'runtime_mean_ms': float(method_all['runtime_ms'].mean()),
                'runtime_median_ms': float(method_all['runtime_ms'].median()),
                'yaw_mae': float(method_success['abs_yaw_error'].mean()) if len(method_success) else np.nan,
                'yaw_median_ae': float(method_success['abs_yaw_error'].median()) if len(method_success) else np.nan,
                'yaw_p90_ae': float(method_success['abs_yaw_error'].quantile(0.90)) if len(method_success) else np.nan,
                'yaw_signed_bias': float(method_success['yaw_error'].mean()) if len(method_success) else np.nan,
            }
        )

    columns = [
        'method',
        'image_count',
        'success_count',
        'failure_count',
        'detection_rate',
        'runtime_mean_ms',
        'runtime_median_ms',
        'yaw_mae',
        'yaw_median_ae',
        'yaw_p90_ae',
        'yaw_signed_bias',
    ]
    if not rows:
        return pd.DataFrame(columns=columns)
    return pd.DataFrame(rows, columns=columns).sort_values('yaw_mae')


def compute_metrics_by_yaw_bucket(ground_truth_df: pd.DataFrame, predictions_df: pd.DataFrame) -> pd.DataFrame:
    success_rows = joined_success_rows(ground_truth_df, predictions_df).copy()
    success_rows['yaw_bucket'] = success_rows['gt_yaw'].map(yaw_bucket)
    rows = []
    for (method, bucket), group in success_rows.groupby(['method', 'yaw_bucket'], observed=True):
        rows.append(
            {
                'method': method,
                'yaw_bucket': bucket,
                'count': len(group),
                'yaw_mae': float(group['abs_yaw_error'].mean()),
                'yaw_median_ae': float(group['abs_yaw_error'].median()),
                'yaw_p90_ae': float(group['abs_yaw_error'].quantile(0.90)),
                'yaw_signed_bias': float(group['yaw_error'].mean()),
            }
        )

    columns = ['method', 'yaw_bucket', 'count', 'yaw_mae', 'yaw_median_ae', 'yaw_p90_ae', 'yaw_signed_bias']
    out = pd.DataFrame(rows, columns=columns)
    bucket_order = {'<=30': 0, '30-60': 1, '>60': 2}
    if not out.empty:
        out['_bucket_order'] = out['yaw_bucket'].map(bucket_order)
        out = out.sort_values(['method', '_bucket_order']).drop(columns=['_bucket_order'])
    return out


def save_metrics(ground_truth_df: pd.DataFrame, predictions_df: pd.DataFrame, outputs_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    metrics_dir = outputs_dir / 'metrics'
    metrics_dir.mkdir(parents=True, exist_ok=True)
    summary = compute_metrics_summary(ground_truth_df, predictions_df)
    by_bucket = compute_metrics_by_yaw_bucket(ground_truth_df, predictions_df)
    summary.to_csv(metrics_dir / 'metrics_summary.csv', index=False)
    by_bucket.to_csv(metrics_dir / 'metrics_by_yaw_bucket.csv', index=False)
    return summary, by_bucket

## 6. Plots, Failure Grids, Worst Cases, and Report

In [7]:
BUCKET_ORDER = {'<=30': 0, '30-60': 1, '>60': 2}


def savefig(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()


def plot_metric_bars(metrics_summary_df: pd.DataFrame, metrics_by_bucket_df: pd.DataFrame, plots_dir: Path) -> None:
    plt.figure(figsize=(8, 4))
    plt.bar(metrics_summary_df['method'], metrics_summary_df['yaw_mae'])
    plt.ylabel('Yaw MAE (degrees)')
    plt.xticks(rotation=25, ha='right')
    savefig(plots_dir / 'yaw_mae_per_method.png')

    plt.figure(figsize=(10, 4))
    for method, group in metrics_by_bucket_df.groupby('method', observed=True):
        ordered = group.copy()
        ordered['_bucket_order'] = ordered['yaw_bucket'].map(BUCKET_ORDER)
        ordered = ordered.sort_values('_bucket_order')
        plt.plot(ordered['yaw_bucket'], ordered['yaw_mae'], marker='o', label=method)
    plt.ylabel('Yaw MAE (degrees)')
    plt.xlabel('Yaw bucket')
    plt.legend()
    savefig(plots_dir / 'yaw_mae_by_bucket.png')

    plt.figure(figsize=(8, 4))
    plt.bar(metrics_summary_df['method'], metrics_summary_df['detection_rate'])
    plt.ylabel('Detection rate')
    plt.xticks(rotation=25, ha='right')
    savefig(plots_dir / 'detection_rate_per_method.png')


def plot_scatter_and_histograms(ground_truth_df: pd.DataFrame, predictions_df: pd.DataFrame, plots_dir: Path) -> None:
    joined = predictions_df.merge(ground_truth_df, on='image_id', how='left')
    success = joined[joined['success']].dropna(subset=['gt_yaw', 'pred_yaw'])

    plt.figure(figsize=(5.5, 5.5))
    for method, group in success.groupby('method'):
        plt.scatter(group['gt_yaw'], group['pred_yaw'], s=10, alpha=0.55, label=method)
    if len(success):
        min_val = min(success['gt_yaw'].min(), success['pred_yaw'].min())
        max_val = max(success['gt_yaw'].max(), success['pred_yaw'].max())
        plt.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1)
    plt.xlabel('GT yaw (degrees)')
    plt.ylabel('Predicted yaw (degrees)')
    plt.legend()
    savefig(plots_dir / 'gt_vs_pred_yaw.png')

    plt.figure(figsize=(9, 4))
    for method, group in success.groupby('method'):
        plt.hist((group['pred_yaw'] - group['gt_yaw']).abs(), bins=40, alpha=0.45, label=method)
    plt.xlabel('Absolute yaw error (degrees)')
    plt.ylabel('Count')
    plt.legend()
    savefig(plots_dir / 'yaw_abs_error_histogram.png')

    plt.figure(figsize=(9, 4))
    for method, group in joined.groupby('method'):
        plt.hist(group['runtime_ms'].dropna(), bins=40, alpha=0.45, label=method)
    plt.xlabel('Runtime (ms)')
    plt.ylabel('Count')
    plt.legend()
    savefig(plots_dir / 'runtime_distribution.png')


def _write_image_grid(rows: pd.DataFrame, output_path: Path, label_fn, max_images: int = 20) -> None:
    rows = rows.head(max_images)
    if rows.empty:
        return
    thumbs = []
    thumb_width = 180
    for record in rows.itertuples(index=False):
        image = cv2.imread(record.image_path)
        if image is None:
            continue
        scale = thumb_width / image.shape[1]
        thumb = cv2.resize(image, (thumb_width, max(1, int(image.shape[0] * scale))))
        cv2.putText(thumb, label_fn(record), (5, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (0, 255, 255), 1, cv2.LINE_AA)
        thumbs.append(thumb)
    if not thumbs:
        return
    cols = 5
    cell_h = max(thumb.shape[0] for thumb in thumbs)
    grid = np.full((int(np.ceil(len(thumbs) / cols)) * cell_h, cols * thumb_width, 3), 255, dtype=np.uint8)
    for idx, thumb in enumerate(thumbs):
        row, col = divmod(idx, cols)
        grid[row * cell_h : row * cell_h + thumb.shape[0], col * thumb_width : col * thumb_width + thumb.shape[1]] = thumb
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output_path), grid)


def save_failure_and_worst_case_grids(
    ground_truth_df: pd.DataFrame,
    predictions_df: pd.DataFrame,
    outputs_dir: Path,
) -> None:
    failure_dir = outputs_dir / 'failure_cases'
    worst_dir = outputs_dir / 'worst_cases'
    for directory in [failure_dir, worst_dir]:
        if directory.exists():
            shutil.rmtree(directory)
        directory.mkdir(parents=True, exist_ok=True)

    joined = predictions_df.merge(ground_truth_df, on='image_id', how='left')
    for method, group in joined.groupby('method', observed=True):
        failed = group[~group['success'].astype(bool)]
        _write_image_grid(
            failed,
            failure_dir / f'{method}_failed_images.jpg',
            lambda row: f"{row.image_id} {row.failure_reason}"[:50],
        )

        success = group[group['success']].dropna(subset=['gt_yaw', 'pred_yaw']).copy()
        if success.empty:
            continue
        success['ae_yaw'] = (success['pred_yaw'] - success['gt_yaw']).abs()
        top_yaw = success.sort_values('ae_yaw', ascending=False)
        top_yaw.to_csv(worst_dir / f'{method}_top_20_worst_yaw.csv', index=False)
        _write_image_grid(
            top_yaw,
            worst_dir / f'{method}_top_20_worst_yaw.jpg',
            lambda row: f"{row.image_id} yaw={row.ae_yaw:.1f}",
        )


def write_short_report(metrics_summary_df: pd.DataFrame, metrics_by_bucket_df: pd.DataFrame, output_path: Path) -> None:
    lines = ['# AFLW2000-3D Cropped Yaw-Only Evaluation', '']
    if not metrics_summary_df.empty:
        best = metrics_summary_df.sort_values('yaw_mae').iloc[0]
        lines.append(
            f"Best method by Yaw MAE: `{best['method']}` with {best['yaw_mae']:.3f} degrees and {best['detection_rate']:.1%} detection rate."
        )
        lines.append('')
        lines.append('## Main Metrics')
        lines.append('```text')
        lines.append(metrics_summary_df.to_string(index=False))
        lines.append('```')
    if not metrics_by_bucket_df.empty:
        lines.extend(['', '## Metrics by Yaw Bucket', '```text', metrics_by_bucket_df.to_string(index=False), '```'])
    output_path.write_text('\n'.join(lines), encoding='utf-8')


def generate_plots_and_reports(
    ground_truth_df: pd.DataFrame,
    predictions_df: pd.DataFrame,
    metrics_summary_df: pd.DataFrame,
    metrics_by_bucket_df: pd.DataFrame,
    outputs_dir: Path,
) -> None:
    outputs_dir.mkdir(parents=True, exist_ok=True)
    plots_dir = outputs_dir / 'plots'
    for directory in [plots_dir]:
        if directory.exists():
            shutil.rmtree(directory)
        directory.mkdir(parents=True, exist_ok=True)

    plot_metric_bars(metrics_summary_df, metrics_by_bucket_df, plots_dir)
    plot_scatter_and_histograms(ground_truth_df, predictions_df, plots_dir)
    save_failure_and_worst_case_grids(ground_truth_df, predictions_df, outputs_dir)
    write_short_report(metrics_summary_df, metrics_by_bucket_df, outputs_dir / 'short_report.md')

## 7. Optional Synthetic Smoke Test

This validates notebook wiring without AFLW2000-3D and without running the real models. It writes to `tmp/head_pose_eval_yaw_smoke`.

In [8]:
RUN_SYNTHETIC_SMOKE = False

if RUN_SYNTHETIC_SMOKE:
    smoke_root = WORKSPACE_ROOT / 'tmp' / 'head_pose_eval_yaw_smoke'
    if smoke_root.exists():
        shutil.rmtree(smoke_root)

    smoke_image_dir = smoke_root / 'datasets' / 'test.data' / 'AFLW2000-3D_crop'
    smoke_config_dir = smoke_root / 'datasets' / 'test.configs'
    smoke_image_dir.mkdir(parents=True, exist_ok=True)
    smoke_config_dir.mkdir(parents=True, exist_ok=True)

    image_ids = ['image00001', 'image00002', 'image00003', 'image00004']
    gt_yaw = np.array([0.0, 18.0, 42.0, 72.0], dtype=np.float32)
    list_lines = []
    for image_id in image_ids:
        image = np.full((96, 96, 3), 210, dtype=np.uint8)
        cv2.circle(image, (48, 48), 22, (90, 120, 160), -1)
        cv2.imwrite(str(smoke_image_dir / f'{image_id}.jpg'), image)
        list_lines.append(f'{image_id}.jpg')

    (smoke_image_dir.parent / 'AFLW2000-3D_crop.list').write_text('\n'.join(list_lines), encoding='utf-8')
    np.save(smoke_config_dir / 'AFLW2000-3D.pose.npy', gt_yaw)

    smoke_gt = build_ground_truth_index(smoke_root, expected_count=4)
    smoke_outputs = smoke_root / 'outputs'
    smoke_outputs.mkdir(parents=True, exist_ok=True)
    smoke_raw_dir = smoke_outputs / 'raw_predictions'
    smoke_raw_dir.mkdir(parents=True, exist_ok=True)

    smoke_rows = []
    for method in ['pyfeat_img2pose', 'mediapipe_matrix']:
        for record in smoke_gt.itertuples(index=False):
            if method == 'mediapipe_matrix' and record.image_id == 'image00004':
                smoke_rows.append(
                    {
                        'image_id': record.image_id,
                        'method': method,
                        'pred_pitch': float('nan'),
                        'pred_yaw': float('nan'),
                        'pred_roll': float('nan'),
                        'success': False,
                        'failure_reason': 'no_face',
                        'runtime_ms': 7.0,
                    }
                )
            else:
                predicted = record.gt_yaw + 2.0 if method == 'pyfeat_img2pose' else -record.gt_yaw + 1.0
                smoke_rows.append(
                    {
                        'image_id': record.image_id,
                        'method': method,
                        'pred_pitch': 0.0,
                        'pred_yaw': float(predicted),
                        'pred_roll': 0.0,
                        'success': True,
                        'failure_reason': None,
                        'runtime_ms': 5.0,
                    }
                )

    smoke_predictions = pd.DataFrame(smoke_rows)
    for method in smoke_predictions['method'].unique():
        smoke_predictions[smoke_predictions['method'] == method].to_csv(smoke_raw_dir / f'{method}_predictions.csv', index=False)

    smoke_normalized, smoke_mapping_config, smoke_candidate_metrics = normalize_predictions(smoke_gt, smoke_raw_dir, smoke_outputs)
    smoke_summary, smoke_by_bucket = save_metrics(smoke_gt, smoke_normalized, smoke_outputs)
    generate_plots_and_reports(smoke_gt, smoke_normalized, smoke_summary, smoke_by_bucket, smoke_outputs)

    display(smoke_gt)
    display(smoke_candidate_metrics)
    display(smoke_summary)
    display(smoke_by_bucket)
    print(json.dumps(smoke_mapping_config, indent=2))

## 8. Run AFLW2000-3D Evaluation

This benchmark will fail clearly if the crop dataset is not present. Set `LIMIT = None` for the full run.

In [10]:
image_dir, list_path, gt_yaw_path = resolve_dataset_layout(DATASET_SEARCH_START)
print('Resolved image_dir:', image_dir)
print('Resolved list_path:', list_path)
print('Resolved gt_yaw_path:', gt_yaw_path)

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
ground_truth_df = build_ground_truth_index(DATASET_SEARCH_START, expected_count=EXPECTED_COUNT)
if LIMIT is not None:
    ground_truth_df = ground_truth_df.head(LIMIT).copy()

ground_truth_df.to_csv(OUTPUTS_DIR / 'ground_truth.csv', index=False)
print('ground_truth rows:', len(ground_truth_df))
display(ground_truth_df.head())
display(ground_truth_df['yaw_bucket'].value_counts().sort_index())

FileNotFoundError: Could not locate the 3DDFA AFLW2000-3D crop benchmark. Expected image crops, list file, and AFLW2000-3D.pose.npy under test.data/test.configs.

In [11]:
raw_predictions_df = run_raw_predictions(ground_truth_df, METHODS, OUTPUTS_DIR, MODEL_PATH)
display(raw_predictions_df.head())

NameError: name 'ground_truth_df' is not defined

In [ ]:
normalized_predictions_df, mapping_config, candidate_metrics_df = normalize_predictions(
    ground_truth_df,
    OUTPUTS_DIR / 'raw_predictions',
    OUTPUTS_DIR,
)

print(json.dumps(mapping_config, indent=2))
display(candidate_metrics_df.head(10))
display(normalized_predictions_df.head())

In [ ]:
metrics_summary_df, metrics_by_bucket_df = save_metrics(ground_truth_df, normalized_predictions_df, OUTPUTS_DIR)
display(metrics_summary_df)
display(metrics_by_bucket_df)

In [ ]:
generate_plots_and_reports(
    ground_truth_df,
    normalized_predictions_df,
    metrics_summary_df,
    metrics_by_bucket_df,
    OUTPUTS_DIR,
)

for output_path in [
    OUTPUTS_DIR / 'ground_truth.csv',
    OUTPUTS_DIR / 'normalized_predictions.csv',
    OUTPUTS_DIR / 'metrics' / 'metrics_summary.csv',
    OUTPUTS_DIR / 'metrics' / 'metrics_by_yaw_bucket.csv',
    OUTPUTS_DIR / 'metrics' / 'convention_mapping.json',
    OUTPUTS_DIR / 'metrics' / 'convention_review.md',
    OUTPUTS_DIR / 'plots' / 'yaw_mae_per_method.png',
    OUTPUTS_DIR / 'plots' / 'gt_vs_pred_yaw.png',
    OUTPUTS_DIR / 'short_report.md',
]:
    print(output_path.relative_to(BENCHMARK_ROOT), output_path.exists())

## 9. Inspect Report

In [ ]:
report_path = OUTPUTS_DIR / 'short_report.md'
if report_path.exists():
    print(report_path.read_text(encoding='utf-8')[:4000])